# AgriTech — 10-image augmentation stress test

I use this Colab notebook to validate the audit inputs, run the full two-model benchmark, or backfill missing outputs. I keep credentials in Colab Secrets; no key values are stored in this notebook. The full-run cell clears the selected output range before it evaluates, while the backfill cell only requests missing prediction cells.

I need the original 10 image files in the configured Google Drive `images/` folder to rerun inference. The packaged image grid is a visualization, not a substitute for those individual inputs.

In [ ]:
# Setup, Drive, workbook/image verification, and secure API-key loading
import os
import openpyxl
from google.colab import drive, userdata

PROJECT_DIR = '/content/drive/MyDrive/AgriTech_Benchmark/data_augmentation'
WORKBOOK_PATH = f'{PROJECT_DIR}/AgriTech_Augmentation_10_Audit_LIGHT.xlsx'
IMAGE_DIR = f'{PROJECT_DIR}/images'
SHEET_NAME = 'Augmentation_10_Audit'

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount('/content/drive')

assert os.path.isdir(PROJECT_DIR), f"Project directory not found: {PROJECT_DIR}"
assert os.path.isdir(IMAGE_DIR), f"Image directory not found: {IMAGE_DIR}"
assert os.path.isfile(WORKBOOK_PATH), f"Workbook not found: {WORKBOOK_PATH}"

expected_headers = [
    "image_id", "filename", "augmentation_applied", "scout_note",
    "gt_boolean", "gt_choice", "gt_score", "gt_spatial",
    "d1_bool_reply", "d1_bool_conf", "jev_bool_reply", "jev_bool_conf",
    "d1_choice_reply", "d1_choice_conf", "jev_choice_reply", "jev_choice_conf",
    "d1_score_reply", "d1_score_conf", "jev_score_reply", "jev_score_conf",
    "d1_spatial_reply", "d1_spatial_conf", "jev_spatial_reply", "jev_spatial_conf",
]
wb_check = openpyxl.load_workbook(WORKBOOK_PATH, read_only=True, data_only=True)
assert SHEET_NAME in wb_check.sheetnames, f"Missing worksheet: {SHEET_NAME}"
ws_check = wb_check[SHEET_NAME]
actual_headers = [ws_check.cell(3, c).value for c in range(1, 25)]
assert actual_headers == expected_headers, f"Unexpected columns 1–24 on row 3: {actual_headers}"

image_records = []
for row in range(4, 14):
    image_id = ws_check.cell(row, 1).value
    filename = str(ws_check.cell(row, 2).value or "").strip()
    assert image_id == row - 3, f"Unexpected image_id at worksheet row {row}: {image_id}"
    assert filename, f"Missing filename at worksheet row {row}"
    image_path = os.path.join(IMAGE_DIR, filename)
    assert os.path.isfile(image_path), f"Image file not found: {image_path}"
    image_records.append((row, filename, image_path))
assert len(image_records) == 10, f"Expected 10 images in rows 4–13; found {len(image_records)}"
wb_check.close()

# Colab Secrets only; never print keys, prefixes, or exception details.
try:
    GEMINI_API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
    OPENROUTER_API_KEY = (userdata.get("OPENROUTER_API_KEY") or "").strip()
except Exception:
    raise RuntimeError("Unable to load required Colab Secrets. Check GEMINI_API_KEY and OPENROUTER_API_KEY in Secrets.") from None
if not GEMINI_API_KEY or not OPENROUTER_API_KEY:
    raise ValueError("GEMINI_API_KEY and OPENROUTER_API_KEY must both be non-empty.")
try:
    OPENROUTER_API_KEY_2 = (userdata.get("OPENROUTER_API_KEY_2") or "").strip()
except Exception:
    OPENROUTER_API_KEY_2 = ""

print(f"Verified {len(image_records)}/10 images on {SHEET_NAME}, rows 4–13; columns I:X are ready. Required API keys loaded securely; optional second OpenRouter key: {'available' if OPENROUTER_API_KEY_2 else 'not set'}.")

In [ ]:
import os, re, json, time, base64, hashlib
import requests
import openpyxl
from google.colab import drive, userdata

# ==============================================================================
# 1. START & END IMAGE
# ==============================================================================
START_IMAGE = 1
END_IMAGE = 10

# ==============================================================================
# 2. PATHS & API KEYS
# ==============================================================================
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/AgriTech_Benchmark/data_augmentation"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
WORKBOOK_PATH = os.path.join(BASE_DIR, "AgriTech_Augmentation_10_Audit_LIGHT.xlsx")
SHEET_NAME = "Augmentation_10_Audit"

def resolve_key(possible_names, paste_here=""):
    if paste_here.strip():
        return paste_here.strip()
    for name in possible_names:
        if name in globals() and globals()[name]:
            return str(globals()[name]).strip()
        if name in os.environ and os.environ[name]:
            return str(os.environ[name]).strip()
        try:
            val = userdata.get(name)
            if val:
                return val.strip()
        except Exception:
            pass
    return ""

GEMINI_API_KEY = resolve_key(["GEMINI_API_KEY", "GOOGLE_API_KEY"], paste_here="")
OPENROUTER_API_KEY = resolve_key(["OPENROUTER_API_KEY", "OPENROUTER_KEY", "JEV_API_KEY"], paste_here="")
OPENROUTER_API_KEY_2 = resolve_key(["OPENROUTER_API_KEY_2"], paste_here="")

if not GEMINI_API_KEY or not OPENROUTER_API_KEY:
    raise ValueError(
        "API Keys not found in memory! Either run your top Setup Cell first, "
        "or paste your keys inside paste_here=\"...\" on lines 39-40."
    )

# ==============================================================================
# 3. MODELS & PROMPTS
# ==============================================================================
GOOGLE_MODEL = "gemini-flash-lite-latest"
JEV_MODEL = "qwen/qwen2.5-vl-72b-instruct"

PRIMITIVES = ["boolean", "choice", "score", "spatial"]
PROMPTS = {
    "boolean": 'Is active foliar disease or pest damage present? JSON only: {"pred":true/false,"conf":0.0}.',
    "choice": 'Choose one of fungal_or_pest_pathology, drought_curled_high_fire_fuel, healthy_normal_senescence, invalid_non_foliage, nutrient_deficiency_chlorosis. JSON only: {"pred":"label","conf":0.0}.',
    "score": 'Score foliar desiccation from 1 to 4. JSON only: {"pred":1,"conf":0.0}.',
    "spatial": 'Choose quadrant top_left, top, top_right, bottom_left, center, bottom_right, bottom, or none; provide [ymin,xmin,ymax,xmax] normalized 0..1000 or null. JSON only: {"pred":"label","conf":0.0,"box":null}.'
}

OUTPUT_COLUMNS = {
    "boolean": (9, 10, 11, 12),
    "choice":  (13, 14, 15, 16),
    "score":   (17, 18, 19, 20),
    "spatial": (21, 22, 23, 24),
}

def parse_json_safely(text):
    clean = re.sub(r"```(?:json)?|```", "", str(text or ""), flags=re.I).strip()
    match = re.search(r"\{.*\}", clean, flags=re.S)
    try:
        return json.loads(match.group(0)) if match else {}
    except (ValueError, TypeError):
        return {}

def encode_image_b64(img_path):
    with open(img_path, "rb") as f:
        raw = f.read()
    img_hash = hashlib.md5(raw).hexdigest()[:8]
    return base64.b64encode(raw).decode("ascii"), img_hash

def call_google_isolated(img_b64, scout_note, primitive_type, max_retries=3):
    prompt = f"Scout context: {scout_note}\n{PROMPTS[primitive_type]}"
    body = {
        "contents": [{"parts": [{"text": prompt}, {"inline_data": {"mime_type": "image/jpeg", "data": img_b64}}]}],
        "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}
    }
    for attempt in range(max_retries):
        try:
            url = f"https://generativelanguage.googleapis.com/v1beta/models/{GOOGLE_MODEL}:generateContent?key={GEMINI_API_KEY}"
            response = requests.post(url, json=body, timeout=90)
            if response.status_code == 429 and attempt < max_retries - 1:
                time.sleep(2 * (attempt + 1))
                continue
            if not response.ok:
                print(f"  [!] Google HTTP {response.status_code} on {primitive_type}: {response.text[:150]}")
                continue
            data = response.json()
            text = data.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")
            return parse_json_safely(text)
        except Exception as e:
            print(f"  [!] Google exception on {primitive_type} (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(1)
    return {}

def call_jev_isolated(img_b64, scout_note, primitive_type, max_retries=3):
    body = {
        "model": JEV_MODEL,
        "temperature": 0,
        "max_tokens": 150,  # Prevents 113k token credit reservation error (HTTP 402)
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": f"Scout context: {scout_note}\n{PROMPTS[primitive_type]}"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
        ]}]
    }
    api_keys = [k for k in [OPENROUTER_API_KEY, OPENROUTER_API_KEY_2] if k]
    for attempt in range(max_retries):
        key_to_use = api_keys[attempt % len(api_keys)]
        headers = {"Authorization": f"Bearer {key_to_use}", "Content-Type": "application/json"}
        try:
            response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=body, timeout=120)
            if response.status_code == 429 and attempt < max_retries - 1:
                time.sleep(3 * (attempt + 1))
                continue
            if not response.ok:
                print(f"  [!] Jev HTTP {response.status_code} on {primitive_type}: {response.text[:180]}")
                continue
            data = response.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return parse_json_safely(text)
        except Exception as e:
            print(f"  [!] Jev exception on {primitive_type} (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(1)
    return {}

# ==============================================================================
# 4. RUNNER LOOP (Wipes target rows I:X first, then writes fresh results)
# ==============================================================================
if not (1 <= START_IMAGE <= END_IMAGE <= 10):
    raise ValueError("START_IMAGE and END_IMAGE must be within 1..10")

wb = openpyxl.load_workbook(WORKBOOK_PATH)
ws = wb[SHEET_NAME]

print(f"Clearing old outputs in rows {START_IMAGE+3} to {END_IMAGE+3}, columns I:X...")
for img_num in range(START_IMAGE, END_IMAGE + 1):
    r = img_num + 3
    for c in range(9, 25):
        ws.cell(r, c, None)
wb.save(WORKBOOK_PATH)
print("Target cells wiped clean and saved.\n" + "="*60)

for img_num in range(START_IMAGE, END_IMAGE + 1):
    row = img_num + 3
    filename = str(ws.cell(row, 2).value or "").strip()
    scout_note = str(ws.cell(row, 4).value or "")
    image_path = os.path.join(IMAGE_DIR, filename)
    if not filename or not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image for worksheet row {row} not found: {image_path}")

    img_b64, img_hash = encode_image_b64(image_path)
    print(f"\n[{img_num:02d}/10] {filename} (MD5: {img_hash}) | Path: {image_path}")

    for primitive in PRIMITIVES:
        d1_parsed = call_google_isolated(img_b64, scout_note, primitive)
        jev_parsed = call_jev_isolated(img_b64, scout_note, primitive)

        d1_reply, d1_conf = d1_parsed.get("pred"), d1_parsed.get("conf")
        jev_reply, jev_conf = jev_parsed.get("pred"), jev_parsed.get("conf")

        print(f"  -> {primitive.upper():7s} | D1: {str(d1_reply):28s} ({d1_conf}) | JEV: {str(jev_reply):28s} ({jev_conf})")

        for col, value in zip(OUTPUT_COLUMNS[primitive], (d1_reply, d1_conf, jev_reply, jev_conf)):
            ws.cell(row, col, value)

    wb.save(WORKBOOK_PATH)
    print(f"[{img_num:02d}/10] {filename}: reply/confidence cells I:X saved.")

print(f"\nCompleted images {START_IMAGE}–{END_IMAGE}; workbook checkpointed at {WORKBOOK_PATH}.")

In [ ]:
import os, re, json, time, base64, hashlib, io
import requests
import openpyxl
from PIL import Image
from google.colab import userdata

# ==============================================================================
# 1. CONFIGURATION & API KEYS
# ==============================================================================
START_IMAGE = 7  # <--- Start at 7 since 4, 5, 6 are already done!
END_IMAGE = 10

GEMINI_API_KEY = (globals().get("GEMINI_API_KEY") or userdata.get("GEMINI_API_KEY") or "").strip()
NEW_OPENROUTER_KEY = (userdata.get("OPENROUTER_API_KEY_3") or "").strip()

assert GEMINI_API_KEY, "Missing GEMINI_API_KEY!"
assert NEW_OPENROUTER_KEY, "Missing OPENROUTER_API_KEY_3 in Colab Secrets!"

BASE_DIR = "/content/drive/MyDrive/AgriTech_Benchmark/data_augmentation"
WORKBOOK_PATH = os.path.join(BASE_DIR, "AgriTech_Augmentation_10_Audit_LIGHT.xlsx")
IMAGE_DIR = os.path.join(BASE_DIR, "images")

GOOGLE_MODEL = "gemini-flash-lite-latest"
JEV_MODEL = "qwen/qwen2.5-vl-72b-instruct"

PRIMITIVES = ["boolean", "choice", "score", "spatial"]
PROMPTS = {
    "boolean": 'Is active foliar disease or pest damage present? JSON only: {"pred":true/false,"conf":0.0}.',
    "choice": 'Choose one of fungal_or_pest_pathology, drought_curled_high_fire_fuel, healthy_normal_senescence, invalid_non_foliage, nutrient_deficiency_chlorosis. JSON only: {"pred":"label","conf":0.0}.',
    "score": 'Score foliar desiccation from 1 to 4. JSON only: {"pred":1,"conf":0.0}.',
    "spatial": 'Choose quadrant top_left, top, top_right, bottom_left, center, bottom_right, bottom, or none; provide [ymin,xmin,ymax,xmax] normalized 0..1000 or null. JSON only: {"pred":"label","conf":0.0,"box":null}.'
}

OUTPUT_COLUMNS = {
    "boolean": (9, 10, 11, 12),
    "choice":  (13, 14, 15, 16),
    "score":   (17, 18, 19, 20),
    "spatial": (21, 22, 23, 24),
}

# FIX 1: Downscale image in memory so it uses ~250 tokens instead of ~1,600 tokens!
def encode_image_b64(path, max_size=512):
    with open(path, "rb") as f:
        raw = f.read()
    img_hash = hashlib.md5(raw).hexdigest()[:8]

    img = Image.open(io.BytesIO(raw)).convert("RGB")
    img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8"), img_hash

def parse_json_response(text):
    if not text:
        return {}
    cleaned = re.sub(r"```(?:json)?|```", "", text).strip()
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass
    return {}

def call_google_isolated(img_b64, scout_note, primitive_type, max_retries=4):
    prompt = f"Scout context: {scout_note}\n{PROMPTS[primitive_type]}"
    body = {
        "contents": [{"parts": [{"text": prompt}, {"inline_data": {"mime_type": "image/jpeg", "data": img_b64}}]}],
        "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}
    }
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GOOGLE_MODEL}:generateContent?key={GEMINI_API_KEY}"
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, json=body, timeout=90)
            if resp.status_code in (429, 500, 503) and attempt < max_retries - 1:
                time.sleep(4 * (attempt + 1))
                continue
            if resp.status_code != 200:
                print(f"  [!] Google HTTP {resp.status_code} on {primitive_type}: {resp.text[:180]}")
                continue
            data = resp.json()
            text = data.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")
            parsed = parse_json_response(text)
            if "pred" in parsed:
                return parsed
        except Exception as e:
            print(f"  [!] Google exception on {primitive_type} (attempt {attempt+1}): {e}")
            time.sleep(3)
    return {"pred": None, "conf": None}

# FIX 2: Stop retrying immediately on HTTP 402 so we don't spam in-flight reservations
def call_jev_single_key(img_b64, scout_note, primitive_type, max_retries=3):
    body = {
        "model": JEV_MODEL,
        "temperature": 0,
        "max_tokens": 45,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": f"Scout context: {scout_note}\n{PROMPTS[primitive_type]}"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
        ]}]
    }
    headers = {"Authorization": f"Bearer {NEW_OPENROUTER_KEY}", "Content-Type": "application/json"}
    for attempt in range(max_retries):
        try:
            response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=body, timeout=120)
            if response.status_code == 402:
                print(f"  [!] Jev HTTP 402 (Credit limit reached) on {primitive_type}: {response.text[:180]}")
                break  # Do not retry 402!
            if response.status_code in (429, 500, 502, 503) and attempt < max_retries - 1:
                time.sleep(3 * (attempt + 1))
                continue
            if response.status_code != 200:
                print(f"  [!] Jev HTTP {response.status_code} on {primitive_type}: {response.text[:180]}")
                time.sleep(2)
                continue
            data = response.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            parsed = parse_json_response(text)
            if "pred" in parsed:
                return parsed
        except Exception as e:
            print(f"  [!] Jev exception on {primitive_type} (attempt {attempt+1}): {e}")
            time.sleep(3)
    return {"pred": None, "conf": None}

# ==============================================================================
# 2. LOAD SHEET & SMART FILL LOOP
# ==============================================================================
wb = openpyxl.load_workbook(WORKBOOK_PATH)

candidates = [s for s in wb.sheetnames if "light" in s.lower() or "audit" in s.lower()]
sheet_to_use = candidates[0] if candidates else wb.sheetnames[0]
ws = wb[sheet_to_use]
print(f"Using worksheet: '{ws.title}'")

for img_num in range(START_IMAGE, END_IMAGE + 1):
    row = img_num + 3
    filename = str(ws.cell(row, 2).value or "").strip()
    scout_note = str(ws.cell(row, 4).value or "")
    image_path = os.path.join(IMAGE_DIR, filename)
    img_b64, img_hash = encode_image_b64(image_path)

    print(f"\n[{img_num:02d}/10] Checking {filename} (MD5: {img_hash})")

    for primitive in PRIMITIVES:
        d1_r_col, d1_c_col, jev_r_col, jev_c_col = OUTPUT_COLUMNS[primitive]

        d1_reply = ws.cell(row, d1_r_col).value
        d1_conf  = ws.cell(row, d1_c_col).value
        jev_reply = ws.cell(row, jev_r_col).value
        jev_conf  = ws.cell(row, jev_c_col).value

        # Only call Google D1 if missing
        if d1_reply is None or d1_conf is None:
            print(f"  -> Filling missing D1 for {primitive}...")
            d1_parsed = call_google_isolated(img_b64, scout_note, primitive)
            d1_reply, d1_conf = d1_parsed.get("pred"), d1_parsed.get("conf")
            ws.cell(row, d1_r_col, d1_reply)
            ws.cell(row, d1_c_col, d1_conf)

        # FIX 3: Removed "or jev_conf == 0" so valid False (0.0) answers aren't re-run!
        if jev_reply is None or jev_conf is None:
            time.sleep(2.0)
            jev_parsed = call_jev_single_key(img_b64, scout_note, primitive)
            jev_reply, jev_conf = jev_parsed.get("pred"), jev_parsed.get("conf")
            ws.cell(row, jev_r_col, jev_reply)
            ws.cell(row, jev_c_col, jev_conf)

        print(f"  -> {primitive.upper():7s} | D1: {str(d1_reply):28s} ({d1_conf}) | JEV: {str(jev_reply):28s} ({jev_conf})")

    wb.save(WORKBOOK_PATH)

print("\nAll missing cells successfully filled and saved to:", WORKBOOK_PATH)